# Attribution des accessions UniProt aux peptides crosslinkés (XlinkX)

Ce notebook prend l'export XlinkX/Proteome Discoverer (colonnes `Sequence A` / `Sequence B`) et ajoute
pour chaque peptide l'accession UniProt de la protéine d'origine (**Accession A**, **Accession B**).
**Aucune information sur l'organisme n'est nécessaire** : elle est déduite des peptides eux-mêmes.

**Méthode**
1. **Identification de l'organisme** : les peptides les plus longs (≥ 15 aa, quasi uniques dans le
   vivant) sont recherchés dans *tout* UniProtKB via **UniProt Peptide Search**. L'organisme qui
   contient le plus de ces peptides est retenu.
2. **Protéome de référence figé** : le protéome de référence UniProt de cet organisme est téléchargé
   en FASTA et enregistré avec le numéro de release UniProt (`reference_proteome_<UP>_<release>.fasta`).
   Ce fichier est la référence de tous les calculs suivants ; en le fournissant via `PROTEOME_FASTA`,
   le notebook est rejoué à l'identique, sans réseau.
3. **Correspondance exacte** de chaque peptide contre ce protéome (Leu = Ile, indiscernables en MS).
4. **Attribution par position** : `Leading Protein Position` = position, dans la protéine, du résidu
   crosslinké, soit `début du peptide + Crosslinker Position − 1`. Seules les protéines qui vérifient
   cette égalité sont candidates. Si plusieurs protéines restent candidates, l'accession est
   **ambiguë** et la ligne est signalée (voir `AMBIGUOUS_ACCESSION`).
5. **Type de lien** : inter-protéine seulement si les candidats de A et de B n'ont aucune protéine en commun.
6. Réponse à la question (section 9) avec le tableau complet des lignes décisives.

> Pourquoi pas un BLAST ? Les peptides font en médiane 8 acides aminés : BLAST n'est pas fiable
> sur des séquences aussi courtes et serait très lent pour ~9 000 peptides. Comme les peptides ont été
> identifiés contre une base de données, une correspondance **exacte** est la bonne approche.

**Exécution** : `Kernel → Restart Kernel and Run All Cells`, puis enregistrer le notebook avec ses sorties.

In [ ]:
# Dépendances (à décommenter si besoin)
# %pip install pandas requests openpyxl tqdm

import json, re, time, bisect
from collections import Counter
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

try:
    from tqdm.auto import tqdm
except ImportError:          # tqdm est optionnel
    def tqdm(x, **k): return x

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 250)

## 1. Paramètres

In [ ]:
INPUT_CSV = "Crosslink Data Anonymized 2.2.csv"   # fichier fourni
OUTPUT_PREFIX = "Crosslink_with_accessions"        # -> .csv et .xlsx

# Référence figée : chemin vers un FASTA UniProt déjà téléchargé (ex. celui produit par la section 5
# lors d'une exécution précédente). None = identification de l'organisme + téléchargement.
PROTEOME_FASTA = None

# Organisme : None = déduit automatiquement des peptides (section 4).
ORGANISM_TAXID = None
N_PROBES = 30            # nombre de peptides longs utilisés pour identifier l'organisme
PROBE_MIN_LEN = 15       # longueur minimale d'un peptide "sonde"
PROBE_MAX_HITS = 200     # sonde ignorée si elle se retrouve dans plus d'entrées (protéine trop conservée)

LEU_EQ_ILE = True        # Leu et Ile considérés comme identiques

# Peptide compatible avec plusieurs protéines à la position indiquée :
#   "exclude"      -> la ligne n'a pas d'accession unique ; elle est exclue du classement (section 10)
#   "alphabetical" -> on retient la première accession par ordre alphabétique
AMBIGUOUS_ACCESSION = "exclude"

CACHE_DIR = Path("cache_uniprot")
PEPSEARCH_URL = "https://peptidesearch.uniprot.org/asyncrest"
UNIPROT_REST = "https://rest.uniprot.org/uniprotkb"
PROTEOMES_REST = "https://rest.uniprot.org/proteomes"

CACHE_DIR.mkdir(exist_ok=True)
session = requests.Session()
session.headers["User-Agent"] = "crosslink-accession-notebook (python-requests)"

## 2. Lecture des données

In [ ]:
if not Path(INPUT_CSV).exists():
    raise FileNotFoundError(f"{INPUT_CSV!r} introuvable dans {Path.cwd()} ; fichiers CSV présents : "
                            f"{sorted(p.name for p in Path.cwd().glob('*.csv'))}")

df = pd.read_csv(INPUT_CSV)
df = df[df["Sequence A"].notna() & df["Sequence B"].notna()].reset_index(drop=True)

def clean(seq):
    # garde uniquement les acides aminés (au cas où des modifs seraient écrites dans la séquence)
    return re.sub(r"[^A-Z]", "", str(seq).upper())

df["_pepA"] = df["Sequence A"].map(clean)
df["_pepB"] = df["Sequence B"].map(clean)
peptides = sorted(set(df["_pepA"]) | set(df["_pepB"]))

print(f"Fichier : {INPUT_CSV}")
print(f"{len(df)} identifications, {df['Sequence'].nunique()} paires uniques, {len(peptides)} peptides uniques")

## 3. Fonctions d'accès à UniProt

In [ ]:
ACC_RE = re.compile(r"\b(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})(?:-\d+)?\b")

def peptide_search(peps, tax_ids=(), max_wait=1800):
    # Soumet un job UniProt Peptide Search et renvoie l'ensemble des accessions trouvées.
    # tax_ids vide = recherche dans tout UniProtKB.
    data = {"peps": ",".join(peps)}
    if tax_ids:
        data["taxIds"] = ",".join(map(str, tax_ids))
    if LEU_EQ_ILE:
        data["lEQi"] = "on"

    for attempt in range(5):
        r = session.post(PEPSEARCH_URL, data=data, allow_redirects=False, timeout=120)
        if r.status_code not in (429, 500, 502, 503, 504):
            break
        time.sleep(2 ** attempt * 5)
    if r.status_code == 200:            # réponse directe
        return set(ACC_RE.findall(r.text))
    if r.status_code not in (201, 202, 303) or "Location" not in r.headers:
        raise RuntimeError(f"Soumission refusée ({r.status_code}): {r.text[:300]}")

    job_url = requests.compat.urljoin(PEPSEARCH_URL + "/", r.headers["Location"])
    t0 = time.time()
    while time.time() - t0 < max_wait:
        r = session.get(job_url, allow_redirects=False, timeout=120)
        if r.status_code == 200:
            return set(ACC_RE.findall(r.text))
        if r.status_code == 303:        # job en cours (ou redirection vers les résultats)
            nxt = r.headers.get("Location")
            if nxt:
                job_url = requests.compat.urljoin(job_url, nxt)
            time.sleep(int(r.headers.get("Retry-After", 5)))
            continue
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(15)
            continue
        raise RuntimeError(f"Erreur job {job_url} ({r.status_code}): {r.text[:300]}")
    raise TimeoutError(f"Job trop long : {job_url}")

def get_with_retry(url, params, tries=5):
    for attempt in range(tries):
        r = session.get(url, params=params, timeout=600)
        if r.ok:
            return r
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 ** attempt * 5)
            continue
        r.raise_for_status()
    r.raise_for_status()

def fetch_organisms(accs, chunk=100):
    # accession -> (taxon ID, nom de l'organisme)
    base = sorted({a.split("-")[0] for a in accs})
    out = {}
    for i in tqdm(range(0, len(base), chunk), desc="Organismes des sondes"):
        r = get_with_retry(f"{UNIPROT_REST}/accessions",
                           {"accessions": ",".join(base[i:i + chunk]),
                            "fields": "accession,organism_id,organism_name", "format": "tsv"})
        t = pd.read_csv(StringIO(r.text), sep="\t", dtype=str).fillna("")
        out.update(zip(t["Entry"], zip(t["Organism (ID)"], t["Organism"])))
    return out

HEADER_RE = re.compile(r"^>(sp|tr)\|([^|]+)\|\S+\s*(.*?)(?:\s+OS=(.*?))?(?:\s+OX=(\d+))?(?:\s+GN=(\S+))?(?:\s+PE=\d.*)?$")

def parse_fasta(text):
    rows, head, seq = [], None, []
    def flush():
        if head is None:
            return
        m = HEADER_RE.match(head)
        if m:
            db, acc, prot, org, ox, gene = m.groups()
        else:
            db, acc, prot, org, ox, gene = "", head[1:].split()[0], "", "", "", ""
        rows.append({"accession": acc, "reviewed": db == "sp", "protein": prot or "",
                     "organism": org or "", "taxid": ox or "", "gene": gene or "",
                     "sequence": "".join(seq)})
    for line in text.splitlines():
        if line.startswith(">"):
            flush()
            head, seq = line.strip(), []
        elif line.strip():
            seq.append(line.strip())
    flush()
    return pd.DataFrame(rows)

def cached_json(name, compute):
    path = CACHE_DIR / name
    if path.exists():
        return json.loads(path.read_text())
    value = compute()
    path.write_text(json.dumps(value))
    return value

## 4. Identification de l'organisme à partir des peptides

Un peptide de 8 acides aminés se retrouve dans des centaines d'organismes, mais un peptide de
15 à 30 acides aminés est pratiquement unique. On recherche donc les peptides les plus longs dans
**tout** UniProtKB (sans filtre d'organisme) et on compte, pour chaque organisme, combien de ces
peptides il contient. Les peptides présents dans trop d'entrées (protéines très conservées :
histones, ubiquitine, protéines ribosomiques…) sont écartés car peu informatifs.

Étape sautée si `PROTEOME_FASTA` ou `ORGANISM_TAXID` est fourni.

In [ ]:
if PROTEOME_FASTA or ORGANISM_TAXID:
    print("Étape sautée (référence ou organisme fourni).")
else:
    probes = sorted((p for p in peptides if len(p) >= PROBE_MIN_LEN), key=lambda p: (-len(p), p))[:N_PROBES]
    print(f"{len(probes)} peptides sondes (longueur {len(probes[-1])}–{len(probes[0])} aa)")

    probe_hits = cached_json("organism_probes.json", lambda: {
        p: sorted(peptide_search([p])) for p in tqdm(probes, desc="Peptide Search (tout UniProtKB)")})

    informative = {p: h for p, h in probe_hits.items() if 0 < len(h) <= PROBE_MAX_HITS}
    print(f"{len(informative)} sondes informatives "
          f"({sum(not h for h in probe_hits.values())} sans résultat, "
          f"{sum(len(h) > PROBE_MAX_HITS for h in probe_hits.values())} trop conservées)")

    acc2org = fetch_organisms({a for h in informative.values() for a in h})
    votes, names = Counter(), {}
    for p, accs in informative.items():
        for taxid, name in {acc2org[a.split("-")[0]] for a in accs if a.split("-")[0] in acc2org}:
            votes[taxid] += 1
            names[taxid] = name

    display(pd.DataFrame([{"Taxon ID": t, "Organisme": names[t], "Sondes retrouvées": n,
                           "% des sondes": round(100 * n / max(len(informative), 1), 1)}
                          for t, n in votes.most_common(10)]))
    if not votes:
        raise RuntimeError("Aucun organisme identifié : vérifier l'accès à Peptide Search ou fixer ORGANISM_TAXID.")
    ORGANISM_TAXID = int(votes.most_common(1)[0][0])
    print(f"Organisme retenu : {names[str(ORGANISM_TAXID)]} (taxon {ORGANISM_TAXID})")

## 5. Protéome de référence figé (FASTA + release UniProt)

Le protéome de référence de l'organisme est téléchargé une seule fois, puis enregistré avec son
identifiant (`UP…`) et le numéro de release UniProt. Les exécutions suivantes réutilisent ce fichier :
le résultat ne change plus quand UniProt est mis à jour. Pour rejouer exactement une analyse,
indiquer ce fichier dans `PROTEOME_FASTA`.

In [ ]:
def find_reference_proteome(taxid):
    r = get_with_retry(f"{PROTEOMES_REST}/search",
                       {"query": f"taxonomy_id:{taxid}", "format": "json", "size": 100})
    rows = []
    for p in r.json().get("results", []):
        ptype = str(p.get("proteomeType", ""))
        rows.append({"id": p.get("id"), "type": ptype,
                     "organism": p.get("taxonomy", {}).get("scientificName", ""),
                     "proteins": int(p.get("proteinCount") or 0)})
    if not rows:
        raise RuntimeError(f"Aucun protéome UniProt pour le taxon {taxid}")
    return max(rows, key=lambda x: ("reference" in x["type"].lower(),
                                    "representative" in x["type"].lower(), x["proteins"]))

if PROTEOME_FASTA:
    fasta_path = Path(PROTEOME_FASTA)
else:
    proteome = cached_json(f"proteome_{ORGANISM_TAXID}.json", lambda: find_reference_proteome(ORGANISM_TAXID))
    print(f"Protéome : {proteome['id']} — {proteome['organism']} — {proteome['type']} — {proteome['proteins']} protéines")
    existing = sorted(Path(".").glob(f"reference_proteome_{proteome['id']}_*.fasta"))
    if existing:
        fasta_path = existing[-1]
    else:
        r = get_with_retry(f"{UNIPROT_REST}/stream", {"query": f"proteome:{proteome['id']}", "format": "fasta"})
        release = r.headers.get("X-UniProt-Release", "release-inconnue")
        fasta_path = Path(f"reference_proteome_{proteome['id']}_{release}.fasta")
        fasta_path.write_text(r.text)

entries = parse_fasta(fasta_path.read_text()).drop_duplicates("accession").reset_index(drop=True)
REFERENCE_LABEL = fasta_path.name
print(f"Référence utilisée : {REFERENCE_LABEL} — {len(entries)} protéines "
      f"({entries['reviewed'].sum()} Swiss-Prot, {(~entries['reviewed']).sum()} TrEMBL)")
print("Organisme(s) :", ", ".join(entries["organism"].value_counts().index[:3]))

## 6. Correspondance exacte peptide → protéine(s)

In [ ]:
def norm(s):
    return s.replace("I", "L") if LEU_EQ_ILE else s

# Toutes les séquences concaténées (séparateur '#') pour une recherche rapide.
starts, parts, pos = [], [], 0
for s in entries["sequence"]:
    starts.append(pos)
    parts.append(norm(s))
    pos += len(s) + 1
big = "#".join(parts)

def find_matches(pep):
    # Renvoie {index_entrée: [positions de début, 1-based]}.
    hits, p = {}, norm(pep)
    i = big.find(p)
    while i != -1:
        k = bisect.bisect_right(starts, i) - 1
        hits.setdefault(k, []).append(i - starts[k] + 1)
        i = big.find(p, i + 1)
    return hits

pep_hits = {pep: find_matches(pep) for pep in tqdm(peptides, desc="Matching")}
n_found = sum(bool(h) for h in pep_hits.values())
print(f"{n_found}/{len(peptides)} peptides retrouvés dans la référence ({len(peptides) - n_found} sans correspondance)")

## 7. Attribution des accessions et type de lien

Pour chaque peptide d'une ligne :
- **candidats** = protéines où le résidu crosslinké (`début + Crosslinker Position − 1`) tombe exactement
  sur `Leading Protein Position` ; si aucune protéine ne vérifie la position, tous les matchs sont candidats
  (statut « position non vérifiée ») ;
- **statut** : `unique` (1 candidat), `ambiguë` (plusieurs candidats), `non trouvé` ;
- **accession** : le candidat unique ; si ambiguë, vide (`AMBIGUOUS_ACCESSION = "exclude"`) ou
  première par ordre alphabétique (`"alphabetical"`).

**Type de lien** : `intra` si A et B ont la même accession unique ; `ambigu (intra possible)` si les
candidats de A et de B ont au moins une protéine en commun ; sinon `inter`.

In [ ]:
E = entries.to_dict("records")

def residue_hits(pep, lead_pos, xl_pos):
    # -> (candidats [(accession, [positions du résidu crosslinké])], position_vérifiée)
    hits = pep_hits.get(pep, {})
    x = int(xl_pos) if not pd.isna(xl_pos) else 1
    allc = {k: [st + x - 1 for st in v] for k, v in hits.items()}
    if not pd.isna(lead_pos):
        posc = {k: r for k, r in allc.items() if int(lead_pos) in r}
        if posc:
            return sorted((E[k]["accession"], r) for k, r in posc.items()), True
    return sorted((E[k]["accession"], r) for k, r in allc.items()), False

acc_info = entries.set_index("accession")[["gene", "protein", "reviewed", "organism"]].to_dict("index")
cache = {}

def annotate(row, side):
    pep = row[f"_pep{side}"]
    key = (pep, row[f"Leading Protein Position {side}"], row[f"Crosslinker Position {side}"])
    if key not in cache:
        cache[key] = residue_hits(*key)
    cands, pos_ok = cache[key]
    accs = [a for a, _ in cands]
    if not accs:
        status, acc = "non trouvé", ""
    elif len(accs) == 1:
        status, acc = "unique", accs[0]
    else:
        status = "ambiguë"
        acc = accs[0] if AMBIGUOUS_ACCESSION == "alphabetical" else ""
    if accs and not pos_ok:
        status += " (position non vérifiée)"
    info = acc_info.get(acc, {})
    return {f"Accession {side}": acc,
            f"Accession status {side}": status,
            f"Candidates {side}": ";".join(accs),
            f"Crosslinked residue {side}": ";".join(f"{a}:{'/'.join(map(str, r))}" for a, r in cands),
            f"Gene {side}": info.get("gene", ""),
            f"Protein {side}": info.get("protein", ""),
            f"Reviewed {side}": ("Swiss-Prot" if info.get("reviewed") else "TrEMBL") if info else ""}

annot = pd.DataFrame([{**annotate(r, "A"), **annotate(r, "B")} for _, r in tqdm(df.iterrows(), total=len(df))])

def link_type(r):
    ca = set(filter(None, r["Candidates A"].split(";")))
    cb = set(filter(None, r["Candidates B"].split(";")))
    if not ca or not cb:
        return "non attribué"
    if ca & cb:
        return "intra" if ca == cb and len(ca) == 1 else "ambigu (intra possible)"
    return "inter"

annot["Link type"] = annot.apply(link_type, axis=1)
annot["Organism"] = [acc_info.get(a, {}).get("organism", "") for a in annot["Accession A"]]

out = pd.concat([df.drop(columns=["_pepA", "_pepB"]), annot], axis=1)
key_cols = ["Sequence", "Sequence A", "Accession A", "Accession status A", "Gene A",
            "Sequence B", "Accession B", "Accession status B", "Gene B", "Link type"]
out = out[key_cols + [c for c in out.columns if c not in key_cols]]
display(out[key_cols].head(10))

## 8. Résumé et export

In [ ]:
display(out["Link type"].value_counts().to_frame("lignes"))
for s in "AB":
    display(out[f"Accession status {s}"].value_counts().to_frame(f"statut peptide {s}"))

peptide_table = pd.DataFrame([
    {"Peptide": p, "N matches": len(h),
     "Matches (accession:début)": ";".join(f"{E[k]['accession']}:{'/'.join(map(str, v))}"
                                           for k, v in sorted(h.items(), key=lambda kv: E[kv[0]]["accession"]))}
    for p, h in pep_hits.items()])

out.to_csv(f"{OUTPUT_PREFIX}.csv", index=False)
with pd.ExcelWriter(f"{OUTPUT_PREFIX}.xlsx", engine="openpyxl") as xw:
    out.to_excel(xw, sheet_name="Crosslinks", index=False)
    peptide_table.to_excel(xw, sheet_name="Peptides", index=False)
    pd.DataFrame({"Référence": [REFERENCE_LABEL], "Leu=Ile": [LEU_EQ_ILE],
                  "Accession ambiguë": [AMBIGUOUS_ACCESSION]}).to_excel(xw, sheet_name="Paramètres", index=False)
print(f"Écrit : {OUTPUT_PREFIX}.csv et {OUTPUT_PREFIX}.xlsx")

## 9. Question : les 3 crosslinks inter-protéines les plus abondants (XlinkX Score > 40)

- chaque ligne est un crosslink indépendant (pas de regroupement entre runs) ;
- `XlinkX Score` strictement > 40 ;
- `Precursor Abundance` = 0 (ou vide) → non quantifié, exclu ;
- inter-protéine = `Link type == "inter"` ;
- lignes dont une accession est ambiguë : exclues si `AMBIGUOUS_ACCESSION = "exclude"` ;
- tri par abondance décroissante, égalités départagées par ordre alphabétique de l'Accession A.

Le **tableau décisif** liste toutes les lignes quantifiées au-dessus du seuil, de la plus abondante à la
3ᵉ retenue (et quelques suivantes), avec la raison de chaque exclusion et les candidats/positions.

In [ ]:
TOP_N = 3
SCORE_MIN = 40

cand = out[(pd.to_numeric(out["XlinkX Score"], errors="coerce") > SCORE_MIN)
           & (pd.to_numeric(out["Precursor Abundance"], errors="coerce") > 0)].copy()
cand["_abund"] = pd.to_numeric(cand["Precursor Abundance"])
cand = cand.sort_values(["_abund", "Accession A"], ascending=[False, True], kind="mergesort")

def decision(r):
    if r["Link type"] != "inter":
        return f"exclu : {r['Link type']}"
    if not r["Accession A"] or not r["Accession B"]:
        return "exclu : accession ambiguë"
    return "retenu"

cand["Décision"] = cand.apply(decision, axis=1)
kept = cand[cand["Décision"] == "retenu"]
top = kept.head(TOP_N)
rank = {idx: str(i) for i, idx in enumerate(top.index, 1)}
cand.insert(0, "Rang", [rank.get(i, "") for i in cand.index])

print(f"Référence : {REFERENCE_LABEL}")
print(f"{len(cand)} lignes quantifiées avec XlinkX Score > {SCORE_MIN} ; {len(kept)} inter-protéines retenues\n")
answer = "; ".join(f"crosslink {i} : {r['Accession A']} - {r['Accession B']}"
                   for i, (_, r) in enumerate(top.iterrows(), 1))
print("RÉPONSE :", answer)

last = cand.index.get_loc(top.index[-1]) if len(top) else 0
decisive = cand.iloc[:last + 6]
display(decisive[["Rang", "Décision", "Sequence", "XlinkX Score", "Precursor Abundance", "File ID",
                  "Accession A", "Accession B", "Link type",
                  "Leading Protein Position A", "Crosslinker Position A", "Crosslinked residue A",
                  "Leading Protein Position B", "Crosslinker Position B", "Crosslinked residue B"]])
decisive.to_csv(f"{OUTPUT_PREFIX}_decisive_rows.csv", index=False)